# Advanced Models

- Goal: compare stronger models against the best baseline from `train.ipynb` using the same split and evaluation logic.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier



thresholds = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]

## 1. Load Prepared Data

- Load the deduplicated dataset that will be used for advanced model experiments.

In [2]:
data_path = "../dataset/creditcard.csv"
df = pd.read_csv(data_path)
print(df.head())
df = df.drop_duplicates()
df.shape

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

(283726, 31)

## 2. Train / Validation / Test Split

- Reuse the same split strategy from `train.ipynb` so model comparisons stay fair.

In [3]:
x=df.drop("Class", axis=1)
y = df["Class"]

X_temp, X_test, y_temp, y_test = train_test_split(
    x,
    y,
    test_size=0.1,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, 
    y_temp, 
    test_size=15/90,
    stratify = y_temp,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Test set shape:", X_test.shape)

print("Overall class distribution:")
print(y.value_counts(normalize=True))

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation class distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Training set shape: (212794, 30)
Validation set shape: (42559, 30)
Test set shape: (28373, 30)
Overall class distribution:
Class
0    0.998333
1    0.001667
Name: proportion, dtype: float64

Train class distribution:
Class
0    0.998332
1    0.001668
Name: proportion, dtype: float64

Validation class distribution:
Class
0    0.998332
1    0.001668
Name: proportion, dtype: float64

Test class distribution:
Class
0    0.998343
1    0.001657
Name: proportion, dtype: float64


## 3. Baseline Reference

- Record the best baseline result from `train.ipynb` so advanced models can be judged against it.

In [4]:
log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state = 42, max_iter = 1000))
])

log_reg_model.fit(X_train, y_train)

y_pred_logreg = log_reg_model.predict(X_val)
y_score_logreg = log_reg_model.predict_proba(X_val)[:, 1]

results = []

for threshold in thresholds:
    y_pred_thresh_logreg = (y_score_logreg >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_logreg).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_logreg, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_logreg, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_logreg, zero_division=0)
    })

logreg_threshold_df = pd.DataFrame(results)
logreg_threshold_df.round(4)

,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,42391,97,12,59,0.3782,0.8310,0.5198
1,0.05,42462,26,15,56,0.6829,0.7887,0.7320
2,0.10,42470,18,17,54,0.7500,0.7606,0.7552
3,0.30,42473,15,23,48,0.7619,0.6761,0.7164
4,0.50,42480,8,27,44,0.8462,0.6197,0.7154
5,0.70,42482,6,32,39,0.8667,0.5493,0.6724
6,0.90,42482,6,36,35,0.8537,0.4930,0.6250
7,0.95,42482,6,40,31,0.8378,0.4366,0.5741
8,0.99,42482,6,43,28,0.8235,0.3944,0.5333


## 4. First Advanced Model - Random Forest Classifier

- Train the first stronger model and keep the training setup simple.

In [5]:
rf_model_dir = "../models/rf_model"
os.makedirs(rf_model_dir, exist_ok=True)

rf_model_path = f"{rf_model_dir}/rf_default_rs42.joblib"
rf_expected_params = {
    "random_state": 42,
    "class_weight": None
}

if os.path.exists(rf_model_path):
    rf_model = joblib.load(rf_model_path)
    for param_name, expected_value in rf_expected_params.items():
        if rf_model.get_params().get(param_name) != expected_value:
            rf_model = RandomForestClassifier(random_state=42)
            rf_model.fit(X_train, y_train)
            joblib.dump(rf_model, rf_model_path)
            break
else:
    rf_model = RandomForestClassifier(random_state=42)
    rf_model.fit(X_train, y_train)
    joblib.dump(rf_model, rf_model_path)

y_pred_rf = rf_model.predict(X_val)
y_score_rf = rf_model.predict_proba(X_val)[:, 1]

In [6]:
rf_model_dir = "../models/rf_model"
os.makedirs(rf_model_dir, exist_ok=True)

weighted_rf_model_path = f"{rf_model_dir}/rf_weighted_balanced_rs42.joblib"
weighted_rf_expected_params = {
    "random_state": 42,
    "class_weight": "balanced"
}

if os.path.exists(weighted_rf_model_path):
    rf_model_weighted = joblib.load(weighted_rf_model_path)
    for param_name, expected_value in weighted_rf_expected_params.items():
        if rf_model_weighted.get_params().get(param_name) != expected_value:
            rf_model_weighted = RandomForestClassifier(
                random_state=42,
                class_weight="balanced"
            )
            rf_model_weighted.fit(X_train, y_train)
            joblib.dump(rf_model_weighted, weighted_rf_model_path)
            break
else:
    rf_model_weighted = RandomForestClassifier(
        random_state = 42,
        class_weight = "balanced"
    )
    rf_model_weighted.fit(X_train, y_train)
    joblib.dump(rf_model_weighted, weighted_rf_model_path)

y_pred_rf_weighted = rf_model_weighted.predict(X_val)
y_score_rf_weighted = rf_model_weighted.predict_proba(X_val)[:, 1]

## 5. First Advanced Model Evaluation

- Evaluate the model using the same validation metrics as before.

In [7]:
results = []

for threshold in thresholds:
    y_pred_thresh_rf = (y_score_rf >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_rf).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_rf, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_rf, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_rf, zero_division=0)
    })

rf_threshold_df = pd.DataFrame(results)
rf_threshold_df.round(4)

,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,41565,923,9,62,0.0629,0.8732,0.1174
1,0.05,42436,52,12,59,0.5315,0.8310,0.6484
2,0.10,42471,17,13,58,0.7733,0.8169,0.7945
3,0.30,42479,9,14,57,0.8636,0.8028,0.8321
4,0.50,42484,4,17,54,0.9310,0.7606,0.8372
5,0.70,42486,2,24,47,0.9592,0.6620,0.7833
6,0.90,42486,2,40,31,0.9394,0.4366,0.5962
7,0.95,42488,0,50,21,1.0000,0.2958,0.4565
8,0.99,42488,0,63,8,1.0000,0.1127,0.2025


In [8]:
results = []

for threshold in thresholds:
    y_pred_thresh_weighted_rf = (y_score_rf_weighted >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_weighted_rf).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_weighted_rf, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_weighted_rf, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_weighted_rf, zero_division=0)
    })

weighted_rf_threshold_df = pd.DataFrame(results)
weighted_rf_threshold_df.round(4)

,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,41601,887,9,62,0.0653,0.8732,0.1216
1,0.05,42460,28,13,58,0.6744,0.8169,0.7389
2,0.10,42475,13,13,58,0.8169,0.8169,0.8169
3,0.30,42481,7,15,56,0.8889,0.7887,0.8358
4,0.50,42483,5,19,52,0.9123,0.7324,0.8125
5,0.70,42485,3,30,41,0.9318,0.5775,0.7130
6,0.90,42487,1,42,29,0.9667,0.4085,0.5743
7,0.95,42488,0,54,17,1.0000,0.2394,0.3864
8,0.99,42488,0,66,5,1.0000,0.0704,0.1316


## 6. Baseline vs Advanced Model Comparison

- Compare the advanced model directly against the best baseline using the same metrics.

In [9]:
comparison_rows = []

print("Baseline PR-AUC:", average_precision_score(y_val, y_score_logreg).round(6))
print("RF PR-AUC:", average_precision_score(y_val, y_score_rf).round(6))
print("Weighted RF PR-AUC:", average_precision_score(y_val, y_score_rf_weighted).round(6))

for threshold in thresholds:
    y_pred_baseline = (y_score_logreg >= threshold).astype(int)
    y_pred_rf_thresh = (y_score_rf >= threshold).astype(int)
    y_pred_rf_weighted_thresh = (y_score_rf_weighted >= threshold).astype(int)

    comparison_rows.append({
        "Threshold": threshold,

        "Baseline Precision": precision_score(y_val, y_pred_baseline, zero_division=0),
        "RF Precision": precision_score(y_val, y_pred_rf_thresh, zero_division=0),
        "Weighted RF Precision": precision_score(y_val, y_pred_rf_weighted_thresh, zero_division=0),

        "Baseline Recall": recall_score(y_val, y_pred_baseline, zero_division=0),
        "RF Recall": recall_score(y_val, y_pred_rf_thresh, zero_division=0),
        "Weighted RF Recall": recall_score(y_val, y_pred_rf_weighted_thresh, zero_division=0),

        "Baseline F1": f1_score(y_val, y_pred_baseline, zero_division=0),
        "RF F1": f1_score(y_val, y_pred_rf_thresh, zero_division=0),
        "Weighted RF F1": f1_score(y_val, y_pred_rf_weighted_thresh, zero_division=0),
    })

model_threshold_comparison_df = pd.DataFrame(comparison_rows)
model_threshold_comparison_df.round(4)

Baseline PR-AUC: 0.722827
RF PR-AUC: 0.802248
Weighted RF PR-AUC: 0.802146


,Threshold,Baseline Precision,RF Precision,Weighted RF Precision,Baseline Recall,RF Recall,Weighted RF Recall,Baseline F1,RF F1,Weighted RF F1
0,0.01,0.3782,0.0629,0.0653,0.8310,0.8732,0.8732,0.5198,0.1174,0.1216
1,0.05,0.6829,0.5315,0.6744,0.7887,0.8310,0.8169,0.7320,0.6484,0.7389
2,0.10,0.7500,0.7733,0.8169,0.7606,0.8169,0.8169,0.7552,0.7945,0.8169
3,0.30,0.7619,0.8636,0.8889,0.6761,0.8028,0.7887,0.7164,0.8321,0.8358
4,0.50,0.8462,0.9310,0.9123,0.6197,0.7606,0.7324,0.7154,0.8372,0.8125
5,0.70,0.8667,0.9592,0.9318,0.5493,0.6620,0.5775,0.6724,0.7833,0.7130
6,0.90,0.8537,0.9394,0.9667,0.4930,0.4366,0.4085,0.6250,0.5962,0.5743
7,0.95,0.8378,1.0000,1.0000,0.4366,0.2958,0.2394,0.5741,0.4565,0.3864
8,0.99,0.8235,1.0000,1.0000,0.3944,0.1127,0.0704,0.5333,0.2025,0.1316


## 7. Observations

- Random Forest performs meaningfully better than the logistic regression baseline across several thresholds.
- The improvement is not marginal; both Random Forest versions achieve much stronger `F1` scores than the baseline at their better thresholds.
- Unweighted Random Forest is especially strong when precision is prioritized.
- Weighted Random Forest is also competitive and improves recall without collapsing precision the way weighted logistic regression did.
- The best unweighted Random Forest result appears at threshold `0.5`, with the highest overall `F1` among the tested unweighted RF settings.
- The best weighted Random Forest result appears around threshold `0.3`, where it gives a strong balance between precision and recall.
- Random Forest is therefore a real step up from the baseline model and is worth tuning further.
- Both RF versions should be kept as serious candidates until tuning determines which one is actually stronger.

## 8. Hyper Parameter Tuning for Random Forest Model

In [10]:
rf_model_dir = "../models/rf_model"
advanced_outputs_dir = "../outputs/advanced_models"
os.makedirs(rf_model_dir, exist_ok=True)
os.makedirs(advanced_outputs_dir, exist_ok=True)

search_path = f"{rf_model_dir}/rf_random_search.pkl"
search_results_path = f"{advanced_outputs_dir}/rf_search_results.csv"

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=30,
    scoring="average_precision",  # PR-AUC
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

if os.path.exists(search_path):
    search = joblib.load(search_path)
else:
    search.fit(X_train, y_train)
    joblib.dump(search, search_path)

results = pd.DataFrame(search.cv_results_)
results.to_csv(
    search_results_path,
    index=False
)

print("Best params:", search.best_params_)
print("Best PR-AUC:", search.best_score_)

Best params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20, 'class_weight': 'balanced'}
Best PR-AUC: 0.8374857624389515


In [11]:
results = pd.DataFrame(search.cv_results_)

results.sort_values(
    by="mean_test_score",
    ascending=False
)[[
    "mean_test_score",
    "params"
]].head(9)

,mean_test_score,params
27,0.837486,"{'n_estimators': 200, 'min_samples_split': 5, ..."
4,0.837303,"{'n_estimators': 300, 'min_samples_split': 5, ..."
6,0.834355,"{'n_estimators': 100, 'min_samples_split': 2, ..."
15,0.833576,"{'n_estimators': 300, 'min_samples_split': 2, ..."
3,0.832857,"{'n_estimators': 100, 'min_samples_split': 5, ..."
16,0.832750,"{'n_estimators': 200, 'min_samples_split': 5, ..."
14,0.832694,"{'n_estimators': 100, 'min_samples_split': 5, ..."
21,0.831889,"{'n_estimators': 100, 'min_samples_split': 2, ..."
12,0.831716,"{'n_estimators': 200, 'min_samples_split': 2, ..."


In [12]:
rf_bestFT_model = search.best_estimator_

y_score_best_rf = rf_bestFT_model.predict_proba(X_val)[:, 1]

print("Tuned RF PR-AUC:", average_precision_score(y_val, y_score_best_rf).round(6))


results = []

for threshold in thresholds:
    y_pred_thresh_best_rf = (y_score_best_rf >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_best_rf).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_best_rf, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_best_rf, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_best_rf, zero_division=0)
    })

best_rf_threshold_df = pd.DataFrame(results)
best_rf_threshold_df.round(4)

Tuned RF PR-AUC: 0.807971


,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,41989,499,9,62,0.1105,0.8732,0.1962
1,0.05,42428,60,11,60,0.5000,0.8451,0.6283
2,0.10,42464,24,13,58,0.7073,0.8169,0.7582
3,0.30,42479,9,14,57,0.8636,0.8028,0.8321
4,0.50,42481,7,17,54,0.8852,0.7606,0.8182
5,0.70,42485,3,26,45,0.9375,0.6338,0.7563
6,0.90,42486,2,34,37,0.9487,0.5211,0.6727
7,0.95,42487,1,45,26,0.9630,0.3662,0.5306
8,0.99,42488,0,67,4,1.0000,0.0563,0.1067


## 9. Compare All Models So Far

- Compare the best candidate from each model setup tested so far.

In [13]:
best_logreg_row = logreg_threshold_df.loc[logreg_threshold_df["F1 Score"].idxmax()]
best_rf_row = rf_threshold_df.loc[rf_threshold_df["F1 Score"].idxmax()]
best_weighted_rf_row = weighted_rf_threshold_df.loc[weighted_rf_threshold_df["F1 Score"].idxmax()]
best_tuned_rf_row = best_rf_threshold_df.loc[best_rf_threshold_df["F1 Score"].idxmax()]

all_models_comparison_df = pd.DataFrame([
    { 
        "Model": "Logistic Regression",
        "Threshold": best_logreg_row["Threshold"],
        "Precision": best_logreg_row["Precision"],
        "Recall": best_logreg_row["Recall"],
        "F1 Score": best_logreg_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_logreg),
        "ROC-AUC": roc_auc_score(y_val, y_score_logreg)
    },
    {
        "Model": "Random Forest",
        "Threshold": best_rf_row["Threshold"],
        "Precision": best_rf_row["Precision"],
        "Recall": best_rf_row["Recall"],
        "F1 Score": best_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_rf),
        "ROC-AUC": roc_auc_score(y_val, y_score_rf)
    },
    {
        "Model": "Weighted Random Forest",
        "Threshold": best_weighted_rf_row["Threshold"],
        "Precision": best_weighted_rf_row["Precision"],
        "Recall": best_weighted_rf_row["Recall"],
        "F1 Score": best_weighted_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_rf_weighted),
        "ROC-AUC": roc_auc_score(y_val, y_score_rf_weighted)
    },
    {
        "Model": "Tuned Random Forest",
        "Threshold": best_tuned_rf_row["Threshold"],
        "Precision": best_tuned_rf_row["Precision"],
        "Recall": best_tuned_rf_row["Recall"],
        "F1 Score": best_tuned_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_rf),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_rf)
    }
])

all_models_comparison_df.round(4)

,Model,Threshold,Precision,Recall,F1 Score,PR-AUC,ROC-AUC
0,Logistic Regression,0.1,0.7500,0.7606,0.7552,0.7228,0.9739
1,Random Forest,0.5,0.9310,0.7606,0.8372,0.8022,0.9348
2,Weighted Random Forest,0.3,0.8889,0.7887,0.8358,0.8021,0.9351
3,Tuned Random Forest,0.3,0.8636,0.8028,0.8321,0.8080,0.9602


In [14]:
all_threshold_results = []

for _, row in logreg_threshold_df.iterrows():
    all_threshold_results.append({
        "Model": "Logistic Regression",
        **row.to_dict()
    })

for _, row in rf_threshold_df.iterrows():
    all_threshold_results.append({
        "Model": "Random Forest",
        **row.to_dict()
    })

for _, row in weighted_rf_threshold_df.iterrows():
    all_threshold_results.append({
        "Model": "Weighted Random Forest",
        **row.to_dict()
    })

for _, row in best_rf_threshold_df.iterrows():
    all_threshold_results.append({
        "Model": "Tuned Random Forest",
        **row.to_dict()
    })

all_threshold_comparison_df = pd.DataFrame(
    all_threshold_results
)

all_threshold_comparison_df.round(4)

,Model,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,Logistic Regression,0.01,42391.0,97.0,12.0,59.0,0.3782,0.8310,0.5198
1,Logistic Regression,0.05,42462.0,26.0,15.0,56.0,0.6829,0.7887,0.7320
2,Logistic Regression,0.10,42470.0,18.0,17.0,54.0,0.7500,0.7606,0.7552
3,Logistic Regression,0.30,42473.0,15.0,23.0,48.0,0.7619,0.6761,0.7164
4,Logistic Regression,0.50,42480.0,8.0,27.0,44.0,0.8462,0.6197,0.7154
5,Logistic Regression,0.70,42482.0,6.0,32.0,39.0,0.8667,0.5493,0.6724
6,Logistic Regression,0.90,42482.0,6.0,36.0,35.0,0.8537,0.4930,0.6250
7,Logistic Regression,0.95,42482.0,6.0,40.0,31.0,0.8378,0.4366,0.5741
8,Logistic Regression,0.99,42482.0,6.0,43.0,28.0,0.8235,0.3944,0.5333
9,Random Forest,0.01,41565.0,923.0,9.0,62.0,0.0629,0.8732,0.1174


In [15]:
all_threshold_comparison_df.pivot_table(
    index="Threshold",
    columns="Model",
    values="F1 Score"
).round(4)

Model,Logistic Regression,Random Forest,Tuned Random Forest,Weighted Random Forest
Threshold,,,,
0.01,0.5198,0.1174,0.1962,0.1216
0.05,0.7320,0.6484,0.6283,0.7389
0.10,0.7552,0.7945,0.7582,0.8169
0.30,0.7164,0.8321,0.8321,0.8358
0.50,0.7154,0.8372,0.8182,0.8125
0.70,0.6724,0.7833,0.7563,0.7130
0.90,0.6250,0.5962,0.6727,0.5743
0.95,0.5741,0.4565,0.5306,0.3864
0.99,0.5333,0.2025,0.1067,0.1316


In [16]:
all_threshold_comparison_df.pivot_table(
    index="Threshold",
    columns="Model",
    values="Precision"
).round(4)

Model,Logistic Regression,Random Forest,Tuned Random Forest,Weighted Random Forest
Threshold,,,,
0.01,0.3782,0.0629,0.1105,0.0653
0.05,0.6829,0.5315,0.5000,0.6744
0.10,0.7500,0.7733,0.7073,0.8169
0.30,0.7619,0.8636,0.8636,0.8889
0.50,0.8462,0.9310,0.8852,0.9123
0.70,0.8667,0.9592,0.9375,0.9318
0.90,0.8537,0.9394,0.9487,0.9667
0.95,0.8378,1.0000,0.9630,1.0000
0.99,0.8235,1.0000,1.0000,1.0000


In [17]:
all_threshold_comparison_df.pivot_table(
    index="Threshold",
    columns="Model",
    values="Recall"
).round(4)

Model,Logistic Regression,Random Forest,Tuned Random Forest,Weighted Random Forest
Threshold,,,,
0.01,0.8310,0.8732,0.8732,0.8732
0.05,0.7887,0.8310,0.8451,0.8169
0.10,0.7606,0.8169,0.8169,0.8169
0.30,0.6761,0.8028,0.8028,0.7887
0.50,0.6197,0.7606,0.7606,0.7324
0.70,0.5493,0.6620,0.6338,0.5775
0.90,0.4930,0.4366,0.5211,0.4085
0.95,0.4366,0.2958,0.3662,0.2394
0.99,0.3944,0.1127,0.0563,0.0704


## Tuned Random Forest Observations

- The tuned Random Forest remains clearly stronger than the logistic regression baseline.
- Hyperparameter tuning improved the Random Forest family further, especially in the stronger threshold range.
- The tuned model is now one of the leading candidates in the notebook and has been compared directly with the best untuned RF variants.
- The tuned RF does not dominate every single threshold, but it improves the overall quality of the RF model family and justifies the tuning effort.
- At this stage, the main competition is no longer logistic regression versus Random Forest; it is between the different RF variants themselves.
- The tuned RF should now be carried forward as a serious final-model candidate.

## 10. Next Model - Gradient Boosting Family

- The next strong model family to try is `XGBoost` or `LightGBM`.
- `CatBoost` can be explored optionally after that.

## XGBoost

In [18]:
# import sys

# !{sys.executable} -m pip install xgboost

In [19]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_val)
y_score_xgb = xgb_model.predict_proba(X_val)[:, 1]

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("Precision:", precision_score(y_val, y_pred_xgb, zero_division=0))
print("Recall:", recall_score(y_val, y_pred_xgb, zero_division=0))
print("F1:", f1_score(y_val, y_pred_xgb, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, y_score_xgb))
print("PR-AUC:", average_precision_score(y_val, y_score_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_xgb))

Precision: 0.8225806451612904
Recall: 0.7183098591549296
F1: 0.7669172932330827
ROC-AUC: 0.9335613568437551
PR-AUC: 0.7092297444734493
Confusion Matrix:
[[42477    11]
 [   20    51]]


In [20]:
print("XGBoost PR-AUC:", average_precision_score(y_val, y_score_xgb).round(6))


results = []

for threshold in thresholds:
    y_pred_thresh_XGB = (y_score_xgb >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_XGB).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_XGB, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_XGB, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_XGB, zero_division=0)
    })

xgb_threshold_df = pd.DataFrame(results)
xgb_threshold_df.round(4)

XGBoost PR-AUC: 0.70923


,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,42438,50,16,55,0.5238,0.7746,0.6250
1,0.05,42466,22,17,54,0.7105,0.7606,0.7347
2,0.10,42466,22,19,52,0.7027,0.7324,0.7172
3,0.30,42475,13,20,51,0.7969,0.7183,0.7556
4,0.50,42477,11,20,51,0.8226,0.7183,0.7669
5,0.70,42478,10,20,51,0.8361,0.7183,0.7727
6,0.90,42478,10,21,50,0.8333,0.7042,0.7634
7,0.95,42478,10,21,50,0.8333,0.7042,0.7634
8,0.99,42478,10,22,49,0.8305,0.6901,0.7538


## XGBoost Observations

- The base XGBoost model is slightly stronger than logistic regression at its best tested threshold when judged by `F1`, but it is not better on `PR-AUC`.
- The best tested XGBoost threshold so far is `0.7`, where it reaches a strong precision level while keeping recall reasonably stable.
- Base XGBoost is still behind the stronger Random Forest results, so it is not the leading model in the notebook right now.
- Fine-tuning XGBoost is reasonable because boosting models are sensitive to hyperparameters like `learning_rate`, `max_depth`, `n_estimators`, `subsample`, and `colsample_bytree`.

## Finetuned XGBoost Model

In [21]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "gamma": [0, 0.1, 0.3, 0.5],
    "scale_pos_weight": [1, 10, 50, 100, 300]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=100,
    scoring="average_precision",  # PR-AUC
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)
print("Best params:", xgb_search.best_params_)
print("Best PR-AUC:", xgb_search.best_score_)

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best params: {'subsample': 0.8, 'scale_pos_weight': 10, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.7}
Best PR-AUC: 0.8458464561794808


In [22]:
best_xgb = xgb_search.best_estimator_

y_score_best_xgb = best_xgb.predict_proba(X_val)[:, 1]

In [23]:
results = []

for threshold in thresholds:
    y_pred_thresh_best_xgb = (y_score_best_xgb >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_best_xgb).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_best_xgb, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_best_xgb, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_best_xgb, zero_division=0)
    })

best_xgb_threshold_df = pd.DataFrame(results)
best_xgb_threshold_df.round(4)

,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,42417,71,11,60,0.4580,0.8451,0.5941
1,0.05,42468,20,13,58,0.7436,0.8169,0.7785
2,0.10,42473,15,14,57,0.7917,0.8028,0.7972
3,0.30,42478,10,14,57,0.8507,0.8028,0.8261
4,0.50,42480,8,14,57,0.8769,0.8028,0.8382
5,0.70,42481,7,16,55,0.8871,0.7746,0.8271
6,0.90,42484,4,19,52,0.9286,0.7324,0.8189
7,0.95,42485,3,21,50,0.9434,0.7042,0.8065
8,0.99,42487,1,25,46,0.9787,0.6479,0.7797


## Comparing all models

In [24]:
best_logreg_row = logreg_threshold_df.loc[
    logreg_threshold_df["F1 Score"].idxmax()
]

best_rf_row = rf_threshold_df.loc[
    rf_threshold_df["F1 Score"].idxmax()
]

best_weighted_rf_row = weighted_rf_threshold_df.loc[
    weighted_rf_threshold_df["F1 Score"].idxmax()
]

best_tuned_rf_row = best_rf_threshold_df.loc[
    best_rf_threshold_df["F1 Score"].idxmax()
]

best_xgb_row = xgb_threshold_df.loc[
    xgb_threshold_df["F1 Score"].idxmax()
]

best_tuned_xgb_row = best_xgb_threshold_df.loc[
    best_xgb_threshold_df["F1 Score"].idxmax()
]

all_models_comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Threshold": best_logreg_row["Threshold"],
        "Precision": best_logreg_row["Precision"],
        "Recall": best_logreg_row["Recall"],
        "F1 Score": best_logreg_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_logreg),
        "ROC-AUC": roc_auc_score(y_val, y_score_logreg)
    },
    {
        "Model": "Random Forest",
        "Threshold": best_rf_row["Threshold"],
        "Precision": best_rf_row["Precision"],
        "Recall": best_rf_row["Recall"],
        "F1 Score": best_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_rf),
        "ROC-AUC": roc_auc_score(y_val, y_score_rf)
    },
    {
        "Model": "Weighted Random Forest",
        "Threshold": best_weighted_rf_row["Threshold"],
        "Precision": best_weighted_rf_row["Precision"],
        "Recall": best_weighted_rf_row["Recall"],
        "F1 Score": best_weighted_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_rf_weighted),
        "ROC-AUC": roc_auc_score(y_val, y_score_rf_weighted)
    },
    {
        "Model": "Tuned Random Forest",
        "Threshold": best_tuned_rf_row["Threshold"],
        "Precision": best_tuned_rf_row["Precision"],
        "Recall": best_tuned_rf_row["Recall"],
        "F1 Score": best_tuned_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_rf),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_rf)
    },
    {
        "Model": "XGBoost",
        "Threshold": best_xgb_row["Threshold"],
        "Precision": best_xgb_row["Precision"],
        "Recall": best_xgb_row["Recall"],
        "F1 Score": best_xgb_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_xgb),
        "ROC-AUC": roc_auc_score(y_val, y_score_xgb)
    },
    {
        "Model": "Tuned XGBoost",
        "Threshold": best_tuned_xgb_row["Threshold"],
        "Precision": best_tuned_xgb_row["Precision"],
        "Recall": best_tuned_xgb_row["Recall"],
        "F1 Score": best_tuned_xgb_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_xgb),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_xgb)
    }
])

all_models_comparison_df.round(4)

,Model,Threshold,Precision,Recall,F1 Score,PR-AUC,ROC-AUC
0,Logistic Regression,0.1,0.7500,0.7606,0.7552,0.7228,0.9739
1,Random Forest,0.5,0.9310,0.7606,0.8372,0.8022,0.9348
2,Weighted Random Forest,0.3,0.8889,0.7887,0.8358,0.8021,0.9351
3,Tuned Random Forest,0.3,0.8636,0.8028,0.8321,0.8080,0.9602
4,XGBoost,0.7,0.8361,0.7183,0.7727,0.7092,0.9336
5,Tuned XGBoost,0.5,0.8769,0.8028,0.8382,0.8309,0.9736


In [25]:
auc_comparison_df = all_models_comparison_df[
    ["Model", "PR-AUC", "ROC-AUC"]
].sort_values(
    by="PR-AUC",
    ascending=False
)

auc_comparison_df.round(4)

,Model,PR-AUC,ROC-AUC
5,Tuned XGBoost,0.8309,0.9736
3,Tuned Random Forest,0.8080,0.9602
1,Random Forest,0.8022,0.9348
2,Weighted Random Forest,0.8021,0.9351
0,Logistic Regression,0.7228,0.9739
4,XGBoost,0.7092,0.9336


## 11. LightGBM


In [26]:
# import sys

# !{sys.executable} -m pip install lightgbm


In [27]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train, y_train)

y_pred_lgbm = lgbm_model.predict(X_val)
y_score_lgbm = lgbm_model.predict_proba(X_val)[:, 1]

print("Precision:", precision_score(y_val, y_pred_lgbm, zero_division=0))
print("Recall:", recall_score(y_val, y_pred_lgbm, zero_division=0))
print("F1:", f1_score(y_val, y_pred_lgbm, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, y_score_lgbm))
print("PR-AUC:", average_precision_score(y_val, y_score_lgbm))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_lgbm))


[LightGBM] [Info] Number of positive: 355, number of negative: 212439
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004206 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 212794, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001668 -> initscore=-6.394292
[LightGBM] [Info] Start training from score -6.394292
Precision: 0.17801047120418848
Recall: 0.4788732394366197
F1: 0.2595419847328244
ROC-AUC: 0.6370410137344497
PR-AUC: 0.15572926308846724
Confusion Matrix:
[[42331   157]
 [   37    34]]


In [28]:
print("LightGBM PR-AUC:", average_precision_score(y_val, y_score_lgbm).round(6))

results = []

for threshold in thresholds:
    y_pred_thresh_lgbm = (y_score_lgbm >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_lgbm).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_lgbm, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_lgbm, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_lgbm, zero_division=0)
    })

lgbm_threshold_df = pd.DataFrame(results)
lgbm_threshold_df.round(4)


LightGBM PR-AUC: 0.155729


,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,42291,197,37,34,0.1472,0.4789,0.2252
1,0.05,42304,184,37,34,0.1560,0.4789,0.2353
2,0.10,42310,178,37,34,0.1604,0.4789,0.2403
3,0.30,42321,167,37,34,0.1692,0.4789,0.2500
4,0.50,42331,157,37,34,0.1780,0.4789,0.2595
5,0.70,42336,152,37,34,0.1828,0.4789,0.2646
6,0.90,42343,145,38,33,0.1854,0.4648,0.2651
7,0.95,42344,144,38,33,0.1864,0.4648,0.2661
8,0.99,42348,140,38,33,0.1908,0.4648,0.2705


## 12. Finetuned LightGBM Model


In [29]:
lgbm = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 5, 10, 20],
    "min_child_samples": [10, 20, 50, 100],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 0.1, 1.0],
    "scale_pos_weight": [1, 10, 50, 100, 300]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_grid,
    n_iter=100,
    scoring="average_precision",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)
print("Best params:", lgbm_search.best_params_)
print("Best PR-AUC:", lgbm_search.best_score_)


Fitting 3 folds for each of 100 candidates, totalling 300 fits


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'subsample': 1.0, 'scale_pos_weight': 1, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'num_leaves': 63, 'n_estimators': 500, 'min_child_samples': 100, 'max_depth': -1, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
Best PR-AUC: 0.8427709808544269


In [30]:
best_lgbm = lgbm_search.best_estimator_

y_score_best_lgbm = best_lgbm.predict_proba(X_val)[:, 1]

print("Tuned LightGBM PR-AUC:", average_precision_score(y_val, y_score_best_lgbm).round(6))

results = []

for threshold in thresholds:
    y_pred_thresh_best_lgbm = (y_score_best_lgbm >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh_best_lgbm).ravel()

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh_best_lgbm, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh_best_lgbm, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh_best_lgbm, zero_division=0)
    })

best_lgbm_threshold_df = pd.DataFrame(results)
best_lgbm_threshold_df.round(4)


Tuned LightGBM PR-AUC: 0.810638


,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score
0,0.01,42469,19,13,58,0.7532,0.8169,0.7838
1,0.05,42478,10,15,56,0.8485,0.7887,0.8175
2,0.10,42481,7,15,56,0.8889,0.7887,0.8358
3,0.30,42483,5,15,56,0.9180,0.7887,0.8485
4,0.50,42484,4,15,56,0.9333,0.7887,0.8550
5,0.70,42484,4,16,55,0.9322,0.7746,0.8462
6,0.90,42484,4,17,54,0.9310,0.7606,0.8372
7,0.95,42484,4,19,52,0.9286,0.7324,0.8189
8,0.99,42486,2,25,46,0.9583,0.6479,0.7731


## Comparing all models including LightGBM


In [31]:
best_logreg_row = logreg_threshold_df.loc[
    logreg_threshold_df["F1 Score"].idxmax()
]

best_rf_row = rf_threshold_df.loc[
    rf_threshold_df["F1 Score"].idxmax()
]

best_weighted_rf_row = weighted_rf_threshold_df.loc[
    weighted_rf_threshold_df["F1 Score"].idxmax()
]

best_tuned_rf_row = best_rf_threshold_df.loc[
    best_rf_threshold_df["F1 Score"].idxmax()
]

best_xgb_row = xgb_threshold_df.loc[
    xgb_threshold_df["F1 Score"].idxmax()
]

best_tuned_xgb_row = best_xgb_threshold_df.loc[
    best_xgb_threshold_df["F1 Score"].idxmax()
]

best_lgbm_row = lgbm_threshold_df.loc[
    lgbm_threshold_df["F1 Score"].idxmax()
]

best_tuned_lgbm_row = best_lgbm_threshold_df.loc[
    best_lgbm_threshold_df["F1 Score"].idxmax()
]

all_models_with_lgbm_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Threshold": best_logreg_row["Threshold"],
        "Precision": best_logreg_row["Precision"],
        "Recall": best_logreg_row["Recall"],
        "F1 Score": best_logreg_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_logreg),
        "ROC-AUC": roc_auc_score(y_val, y_score_logreg)
    },
    {
        "Model": "Random Forest",
        "Threshold": best_rf_row["Threshold"],
        "Precision": best_rf_row["Precision"],
        "Recall": best_rf_row["Recall"],
        "F1 Score": best_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_rf),
        "ROC-AUC": roc_auc_score(y_val, y_score_rf)
    },
    {
        "Model": "Weighted Random Forest",
        "Threshold": best_weighted_rf_row["Threshold"],
        "Precision": best_weighted_rf_row["Precision"],
        "Recall": best_weighted_rf_row["Recall"],
        "F1 Score": best_weighted_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_rf_weighted),
        "ROC-AUC": roc_auc_score(y_val, y_score_rf_weighted)
    },
    {
        "Model": "Tuned Random Forest",
        "Threshold": best_tuned_rf_row["Threshold"],
        "Precision": best_tuned_rf_row["Precision"],
        "Recall": best_tuned_rf_row["Recall"],
        "F1 Score": best_tuned_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_rf),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_rf)
    },
    {
        "Model": "XGBoost",
        "Threshold": best_xgb_row["Threshold"],
        "Precision": best_xgb_row["Precision"],
        "Recall": best_xgb_row["Recall"],
        "F1 Score": best_xgb_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_xgb),
        "ROC-AUC": roc_auc_score(y_val, y_score_xgb)
    },
    {
        "Model": "Tuned XGBoost",
        "Threshold": best_tuned_xgb_row["Threshold"],
        "Precision": best_tuned_xgb_row["Precision"],
        "Recall": best_tuned_xgb_row["Recall"],
        "F1 Score": best_tuned_xgb_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_xgb),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_xgb)
    },
    {
        "Model": "LightGBM",
        "Threshold": best_lgbm_row["Threshold"],
        "Precision": best_lgbm_row["Precision"],
        "Recall": best_lgbm_row["Recall"],
        "F1 Score": best_lgbm_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_lgbm),
        "ROC-AUC": roc_auc_score(y_val, y_score_lgbm)
    },
    {
        "Model": "Tuned LightGBM",
        "Threshold": best_tuned_lgbm_row["Threshold"],
        "Precision": best_tuned_lgbm_row["Precision"],
        "Recall": best_tuned_lgbm_row["Recall"],
        "F1 Score": best_tuned_lgbm_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_lgbm),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_lgbm)
    }
])

all_models_with_lgbm_df.round(4)


,Model,Threshold,Precision,Recall,F1 Score,PR-AUC,ROC-AUC
0,Logistic Regression,0.10,0.7500,0.7606,0.7552,0.7228,0.9739
1,Random Forest,0.50,0.9310,0.7606,0.8372,0.8022,0.9348
2,Weighted Random Forest,0.30,0.8889,0.7887,0.8358,0.8021,0.9351
3,Tuned Random Forest,0.30,0.8636,0.8028,0.8321,0.8080,0.9602
4,XGBoost,0.70,0.8361,0.7183,0.7727,0.7092,0.9336
5,Tuned XGBoost,0.50,0.8769,0.8028,0.8382,0.8309,0.9736
6,LightGBM,0.99,0.1908,0.4648,0.2705,0.1557,0.6370
7,Tuned LightGBM,0.50,0.9333,0.7887,0.8550,0.8106,0.9746


In [32]:
all_models_with_lgbm_df[
    ["Model", "PR-AUC", "ROC-AUC"]
].sort_values(
    by="PR-AUC",
    ascending=False
).round(4)


,Model,PR-AUC,ROC-AUC
5,Tuned XGBoost,0.8309,0.9736
7,Tuned LightGBM,0.8106,0.9746
3,Tuned Random Forest,0.8080,0.9602
1,Random Forest,0.8022,0.9348
2,Weighted Random Forest,0.8021,0.9351
0,Logistic Regression,0.7228,0.9739
4,XGBoost,0.7092,0.9336
6,LightGBM,0.1557,0.6370


## Final Observations

- `Tuned XGBoost` is the strongest overall model in this notebook by the primary metric, with `PR-AUC = 0.8309`.
- `Tuned LightGBM` is also very strong and gives the best threshold-based `F1 Score` among the tested models, but it still trails `Tuned XGBoost` on `PR-AUC`.
- `Tuned Random Forest` remains competitive and is still a valid backup candidate, but the leading competition is now within the boosted-model family.
- The base `LightGBM` result was very weak, which is a useful reminder that strong model families can still perform poorly without the right hyperparameters.
- At this point, the major useful model families for this project have been covered: linear baseline, simple tree, bagging, and boosting.
- This is a reasonable place to stop adding more models and move toward final model selection and test-set evaluation.
- The most sensible finalists to carry forward are `Tuned XGBoost`, `Tuned LightGBM`, and the best `Random Forest` variant.


## Finalists for Evaluation

- Freeze the models below as the candidates to carry into `evaluate.ipynb`.


In [33]:
finalists_for_evaluation_df = pd.DataFrame([
    {
        "Model": "Tuned XGBoost",
        "Threshold": best_tuned_xgb_row["Threshold"],
        "Precision": best_tuned_xgb_row["Precision"],
        "Recall": best_tuned_xgb_row["Recall"],
        "F1 Score": best_tuned_xgb_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_xgb),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_xgb),
        "Selection Reason": "Best PR-AUC overall",
        "Key Hyperparameters": str(xgb_search.best_params_)
    },
    {
        "Model": "Tuned LightGBM",
        "Threshold": best_tuned_lgbm_row["Threshold"],
        "Precision": best_tuned_lgbm_row["Precision"],
        "Recall": best_tuned_lgbm_row["Recall"],
        "F1 Score": best_tuned_lgbm_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_lgbm),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_lgbm),
        "Selection Reason": "Best F1 overall and second-best PR-AUC",
        "Key Hyperparameters": str(lgbm_search.best_params_)
    },
    {
        "Model": "Tuned Random Forest",
        "Threshold": best_tuned_rf_row["Threshold"],
        "Precision": best_tuned_rf_row["Precision"],
        "Recall": best_tuned_rf_row["Recall"],
        "F1 Score": best_tuned_rf_row["F1 Score"],
        "PR-AUC": average_precision_score(y_val, y_score_best_rf),
        "ROC-AUC": roc_auc_score(y_val, y_score_best_rf),
        "Selection Reason": "Best Random Forest family candidate",
        "Key Hyperparameters": str(search.best_params_)
    }
]).sort_values(by="PR-AUC", ascending=False)

finalists_for_evaluation_df.round(4)


,Model,Threshold,Precision,Recall,F1 Score,PR-AUC,ROC-AUC,Selection Reason,Key Hyperparameters
0,Tuned XGBoost,0.5,0.8769,0.8028,0.8382,0.8309,0.9736,Best PR-AUC overall,"{'subsample': 0.8, 'scale_pos_weight': 10, 'n_..."
1,Tuned LightGBM,0.5,0.9333,0.7887,0.8550,0.8106,0.9746,Best F1 overall and second-best PR-AUC,"{'subsample': 1.0, 'scale_pos_weight': 1, 'reg..."
2,Tuned Random Forest,0.3,0.8636,0.8028,0.8321,0.8080,0.9602,Best Random Forest family candidate,"{'n_estimators': 200, 'min_samples_split': 5, ..."
